# SUN397 TACS visual ViT L16 training and complete web export

Notebook 1 of the SUN397 web deployment series. Train the **image-only TACS** model, evaluate the best validation checkpoint, and export a verified local web bundle. **DINOv3 ViT-L/16 only, 1024 dimensions, 384 pixels, K=1.** No S/16 checkpoint is loaded or converted. No captions, CLIP, Florence, text gates or context-label prior are used.

The visual core follows TACS: one shared backbone for selector/task, CLS dot-product utility over a fixed 20% training pool, patch concatenation before the task transformer blocks, hard Gumbel straight-through task gradients plus categorical policy reward. This is a declared L/16 adaptation, not exact reproduction of the paper's S/16, 256px, 3-seed results. Here: SUN partition 01; stratified 10% validation held out from official train; seed 42; 100 epochs by default; minimum validation CE checkpoint. Batch sizes adapt to VRAM, with gradient accumulation to 64. A detached candidate-feature bank is refreshed at epoch start and again before validation.

Attach **SUN397 images + ClassName.txt + Training_01.txt + Testing_01.txt**, **local DINOv3 repository with hubconf.py and license files**, and **DINOv3 ViT-L/16 pretrained .pth**. No project Python files need to be uploaded: the runtime/training source is embedded below. No pip installation or Internet request is made by notebook cells. Configure the paths in section 2 before Run All.

The final ZIP contains full best weights, exact class order, all pool images, pool manifest, checkpoint-bound feature bank, vendored DINOv3 code, preprocessing, inference CLI/Python adapter, environment, metrics, predictions, SHA256 checks and example real attention outputs. It is usable without the original Kaggle Inputs after extraction, in a compatible Python/PyTorch environment. The current FastAPI web still needs to import this adapter; this notebook does not silently modify the web.

**Attention is computed by the trained backbone, not a separate model.** Query map: last-block classifier CLS attention to query patches while receiving context. Candidate map: last-block selector encoder CLS attention to the selected image. Mean over heads; raw grids and display overlays are included. These maps are not Grad-CAM, not class-conditioned causal explanations, and not a claim to reproduce every Figure 4 visualization detail. The no-context comparison model is not separately trained here.

The export cell refuses incomplete training, a changed best checkpoint, mismatched source/config/pool, failed cold offline reload, changed predictions when attention is enabled, or failed logits parity. It never fills fake predictions or heatmaps. Keep `last.pt` plus this run's `best.pt` to resume; do not alter the epoch horizon or pool under the same output directory. Test is run after training; no test-based selection. The final export is a private deployment artifact subject to original model/dataset license terms.

References: [TACS](https://arxiv.org/abs/2512.00489), [DINOv3 official source](https://github.com/facebookresearch/dinov3).


## 1 Offline environment check
Set tokenizer parallelism before any tokenizer or DataLoader worker. This visual notebook does not import Transformers.


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import importlib.util
import sys
from pathlib import Path
if sys.version_info < (3, 10):
    raise RuntimeError("Python >=3.10 is required")
missing = [name for name in ("torch", "numpy", "PIL", "tqdm") if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError("Missing packages: " + ", ".join(missing) + ". Attach compatible offline wheels or prepare the environment first.")
import torch
if not hasattr(torch.nn.functional, "scaled_dot_product_attention"):
    raise RuntimeError("Use a compatible recent PyTorch with scaled_dot_product_attention")
if not hasattr(torch, "amp") or not hasattr(torch.amp, "GradScaler"):
    raise RuntimeError("Use a recent GPU PyTorch with torch.amp.GradScaler support")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before this full ViT-L/16 training run")
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))


## 2 Paths and training budget

Change paths here when your Kaggle Input owner/slug differs. `DINOV3_WEIGHTS` may be one exact .pth file or a folder containing exactly one ViT-L/16 .pth. Use the official LVD1689M `dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth` with its original filename, not S/16, B/16, SAT or L/16-plus. Set EPOCHS before the first run; changing 100 to 50 during resume is rejected because it changes the scheduler and run identity.

Defaults require about 3.2 GB CPU RAM for pool pixels plus RAM/VRAM for the L/16 model, optimizer and workers. Exact memory and runtime depend on the device. Start with the VRAM batch policy below and reduce TRAIN_BATCH_SIZE before starting a new run if necessary; keep it at least 2 for within-batch policy advantage. Export also needs free disk for best weights, pool images and ZIP. No automatic change of backbone or image size occurs.


In [ ]:
DATASET_ROOT = Path(os.getenv("SUN397_ROOT", "/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397"))
TRAIN_PARTITION = Path(os.getenv("SUN397_TRAIN_PARTITION", "/kaggle/input/datasets/hiuphmnhtrung/sun397/sun397_other/Training_01.txt"))
TEST_PARTITION = Path(os.getenv("SUN397_TEST_PARTITION", "/kaggle/input/datasets/hiuphmnhtrung/sun397/sun397_other/Testing_01.txt"))
DINOV3_REPO_DIR = Path(os.getenv("DINOV3_REPO_DIR", "/kaggle/input/datasets/nguyenducxu/dinov3-code/dinov3-main"))
DINOV3_WEIGHTS = Path(os.getenv("DINOV3_WEIGHTS", "/kaggle/input/datasets/sinhnguync/dinov3-vitl16/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth"))
PROJECT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_DIR = Path(os.getenv("SUN397_TACS_WEB_OUTPUT", str(PROJECT_DIR / "outputs/sun397_tacs_visual_vitl16_web_v1")))
DEVICE = torch.device("cuda")
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
TRAIN_BATCH_SIZE = int(os.getenv("SUN397_TACS_WEB_BATCH", "8" if GPU_GIB >= 80 else "4" if GPU_GIB >= 40 else "2"))
EPOCHS = int(os.getenv("SUN397_TACS_WEB_EPOCHS", "100"))
SEED = 42
IMAGE_SIZE = 384
EFFECTIVE_BATCH_SIZE = 64
NUM_WORKERS = int(os.getenv("SUN397_TACS_WEB_WORKERS", "0"))
RUN_TRAINING = True
RUN_FINAL_TEST = True
RUN_EXPORT = True
print("Output:", OUTPUT_DIR, "| train batch:", TRAIN_BATCH_SIZE, "| epochs:", EPOCHS)
print("Original paper: S/16, 256. This declared deployment adaptation: L/16, 384.")


## 3 Materialize the embedded runtime
Only this notebook is needed on Kaggle. The same model/inference source is copied into the final bundle.


In [ ]:
import hashlib
import json
RUNTIME_DIR = OUTPUT_DIR / "embedded_runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_SOURCES = {'tacs_visual_web.py': '"""Portable visual TACS model and inference contract for the SUN397 web bundle.\n\nNo captions, CLIP, label prior or external service is used. The same backbone\nencodes selector images and the concatenated query/context patch sequence.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport sys\n\nimport numpy as np\nfrom PIL import Image, ImageOps\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\nSCHEMA = "sun397_tacs_visual_web_v1"\nATTENTION_METHOD = "last_block_cls_to_patch_attention_mean_heads"\n\n\ndef file_sha256(path):\n    with Path(path).open("rb") as handle:\n        return hashlib.file_digest(handle, "sha256").hexdigest() if hasattr(hashlib, "file_digest") else _hash_stream(handle)\n\n\ndef _hash_stream(handle):\n    digest = hashlib.sha256()\n    for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):\n        digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef bundle_path(root, relative):\n    relative = str(relative)\n    pure = PurePosixPath(relative)\n    if pure.is_absolute() or "\\\\" in relative or ":" in relative or ".." in pure.parts:\n        raise ValueError(f"Unsafe bundle-relative path: {relative}")\n    root = Path(root).resolve()\n    target = root.joinpath(*pure.parts).resolve()\n    if not target.is_relative_to(root):\n        raise ValueError(f"Bundle path escapes root: {relative}")\n    return target\n\n\ndef read_rgb(source):\n    if isinstance(source, Image.Image):\n        return ImageOps.exif_transpose(source).convert("RGB")\n    if isinstance(source, bytes):\n        source = io.BytesIO(source)\n    with Image.open(source) as image:\n        return ImageOps.exif_transpose(image).convert("RGB")\n\n\ndef preprocess(image, config):\n    """The exported evaluation transform, including EXIF and resize semantics."""\n    image = read_rgb(image)\n    size = int(config["image_size"])\n    image = image.resize((size, size), Image.Resampling.BICUBIC)\n    pixels = torch.from_numpy(np.array(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0\n    mean = torch.tensor(config["mean"], dtype=torch.float32)[:, None, None]\n    std = torch.tensor(config["std"], dtype=torch.float32)[:, None, None]\n    return (pixels - mean) / std\n\n\ndef concatenate_rope(rope):\n    """Token order is [query patches, context patches], not interleaved."""\n    if rope is None:\n        return None\n    if not isinstance(rope, (tuple, list)) or len(rope) != 2:\n        raise TypeError("Expected DINOv3 RoPE (sin, cos); refusing a positional fallback")\n    return tuple(torch.cat((part, part), dim=-2) for part in rope)\n\n\ndef mask_self(scores, excluded_positions=None):\n    if scores.ndim != 2 or scores.shape[1] < 2:\n        raise ValueError("TACS requires at least two candidates")\n    if excluded_positions is not None:\n        positions = torch.as_tensor(excluded_positions, dtype=torch.long, device=scores.device)\n        if positions.shape != (scores.shape[0],) or (positions >= scores.shape[1]).any():\n            raise ValueError("Invalid query-to-candidate mapping")\n        rows = torch.arange(len(positions), device=scores.device)\n        valid = positions >= 0\n        scores = scores.clone()\n        scores[rows[valid], positions[valid]] = -torch.inf\n    if not torch.isfinite(scores).any(dim=1).all():\n        raise ValueError("No eligible candidate for a query")\n    return scores\n\n\nclass _StraightThroughImage(torch.autograd.Function):\n    """Exact derivative of hard Gumbel weights @ fixed pixels, with CPU banking."""\n\n    @staticmethod\n    def forward(ctx, weights, images, chunk_size):\n        if images.ndim != 4 or weights.shape[1] != len(images):\n            raise ValueError("Image bank/selection shape mismatch")\n        hard = F.one_hot(weights.argmax(1), len(images)).to(weights.dtype)\n        if not torch.equal(weights.detach(), hard):\n            raise ValueError("Straight-through image forward requires hard one-hot values")\n        ctx.save_for_backward(images)\n        ctx.chunk_size = int(chunk_size)\n        ctx.weight_device, ctx.weight_dtype = weights.device, weights.dtype\n        indices = weights.argmax(1).to(images.device)\n        return images.index_select(0, indices).to(weights.device, dtype=torch.float32)\n\n    @staticmethod\n    def backward(ctx, grad_images):\n        (bank,) = ctx.saved_tensors\n        grad = grad_images.float().flatten(1)\n        chunks = []\n        # Float32 avoids an fp16 bank matmul silently changing selector gradients.\n        with torch.autocast(device_type=grad.device.type, enabled=False):\n            for start in range(0, len(bank), ctx.chunk_size):\n                block = bank[start:start + ctx.chunk_size].to(grad.device, dtype=torch.float32)\n                chunks.append(grad @ block.flatten(1).T)\n        return torch.cat(chunks, dim=1).to(ctx.weight_device, ctx.weight_dtype), None, None\n\n\ndef straight_through_images(weights, bank, chunk_size=32):\n    return _StraightThroughImage.apply(weights, bank, chunk_size)\n\n\nclass VisualTACS(nn.Module):\n    def __init__(self, backbone, config):\n        super().__init__()\n        self.backbone = backbone\n        self.config = dict(config)\n        dim = int(getattr(backbone, "embed_dim", config["feature_dim"]))\n        if dim != int(config["feature_dim"]):\n            raise ValueError("Backbone feature dimension does not match run configuration")\n        self.head = nn.Sequential(nn.Dropout(float(config.get("head_dropout", 0.1))), nn.Linear(dim, int(config["num_classes"])))\n        self.gradient_checkpointing = bool(config.get("gradient_checkpointing", True))\n\n    @property\n    def prefix_count(self):\n        return 1 + int(getattr(self.backbone, "n_storage_tokens", 0))\n\n    def _tokens(self, images):\n        prepared = self.backbone.prepare_tokens_with_masks(images, None)\n        if not isinstance(prepared, tuple) or len(prepared) != 2:\n            raise TypeError("This runtime requires DINOv3 prepare_tokens_with_masks -> (tokens, grid)")\n        tokens, grid = prepared\n        height, width = map(int, grid)\n        if tokens.shape[1] != self.prefix_count + height * width:\n            raise ValueError("Unexpected prefix/patch token count")\n        rope = self.backbone.rope_embed(H=height, W=width) if self.backbone.rope_embed is not None and not self.training else None\n        return tokens, rope, (height, width)\n\n    @staticmethod\n    def _cls_attention(block, tokens, rope):\n        attention = block.attn\n        qkv = attention.qkv(block.norm1(tokens))\n        batch, count, triple_dim = qkv.shape\n        dim = triple_dim // 3\n        heads = int(attention.num_heads)\n        q, k, _ = qkv.reshape(batch, count, 3, heads, dim // heads).unbind(2)\n        q, k = q.transpose(1, 2), k.transpose(1, 2)\n        if rope is not None:\n            q, k = attention.apply_rope(q, k, rope)\n        weights = ((q[:, :, :1].float() @ k.float().transpose(-2, -1)) * (dim // heads) ** -0.5).softmax(dim=-1)\n        return weights.mean(dim=1)[:, 0]\n\n    def _encode(self, tokens, rope, return_attention=False, grid=None, paired=False):\n        if return_attention and self.training:\n            raise RuntimeError("Attention export is evaluation-only")\n        cls_attention = None\n        for index, block in enumerate(self.backbone.blocks):\n            block_rope = rope\n            if self.training and self.backbone.rope_embed is not None:\n                # Native DINOv3 resamples positional augmentation per block.\n                # Pass that sample into checkpoint so backward reuses it exactly.\n                block_rope = self.backbone.rope_embed(H=grid[0], W=grid[1])\n                if paired:\n                    block_rope = concatenate_rope(block_rope)\n            if return_attention and index == len(self.backbone.blocks) - 1:\n                cls_attention = self._cls_attention(block, tokens, block_rope)\n            if self.training and self.gradient_checkpointing and torch.is_grad_enabled():\n                tokens = checkpoint(block, tokens, block_rope, use_reentrant=False, preserve_rng_state=True)\n            else:\n                tokens = block(tokens, block_rope)\n        if getattr(self.backbone, "untie_cls_and_patch_norms", False):\n            cls = self.backbone.cls_norm(tokens[:, :self.prefix_count])[:, 0]\n        else:\n            cls = self.backbone.norm(tokens)[:, 0]\n        return cls, cls_attention\n\n    def encode_selector(self, images, return_attention=False):\n        tokens, rope, grid = self._tokens(images)\n        features, attention = self._encode(tokens, rope, return_attention, grid=grid)\n        if return_attention:\n            return features, attention[:, self.prefix_count:].reshape(-1, *grid)\n        return features\n\n    def score(self, query_images, candidate_features, excluded_positions=None):\n        query = self.encode_selector(query_images).float()\n        bank = candidate_features.to(query.device, dtype=torch.float32)\n        # Eq. 1 utility: shared CLS dot product. No class labels enter scoring.\n        with torch.autocast(device_type=query.device.type, enabled=False):\n            scores = query @ bank.T\n        return mask_self(scores, excluded_positions), query\n\n    def forward(self, query_images, context_images=None, return_attention=False):\n        tokens, rope, grid = self._tokens(query_images)\n        patch_count = grid[0] * grid[1]\n        if context_images is not None:\n            context_tokens, _, context_grid = self._tokens(context_images)\n            if grid != context_grid:\n                raise ValueError("Query and context must have the same patch grid")\n            tokens = torch.cat((tokens, context_tokens[:, self.prefix_count:]), dim=1)\n            rope = concatenate_rope(rope)\n        features, attention = self._encode(tokens, rope, return_attention, grid=grid, paired=context_images is not None)\n        logits = self.head(features)\n        if not return_attention:\n            return logits\n        result = {"query_classifier": attention[:, self.prefix_count:self.prefix_count + patch_count].reshape(-1, *grid)}\n        if context_images is not None:\n            result["context_classifier"] = attention[:, self.prefix_count + patch_count:].reshape(-1, *grid)\n        return logits, result\n\n\ndef build_model(repo_dir, config, weights_path=None):\n    if config["backbone"] != "dinov3_vitl16" or int(config["feature_dim"]) != 1024:\n        raise ValueError("Production bundle requires DINOv3 ViT-L/16, 1024 dimensions; no S/B substitution")\n    repo = Path(repo_dir).resolve()\n    if not (repo / "dinov3/hub/backbones.py").is_file():\n        raise FileNotFoundError(f"DINOv3 source/dinov3/hub/backbones.py missing: {repo}")\n    kwargs = {"pretrained": weights_path is not None}\n    if weights_path is not None:\n        weights = Path(weights_path).resolve()\n        if not weights.is_file():\n            raise FileNotFoundError(weights)\n        if not weights.name.lower().endswith("lvd1689m-8aa4cbdd.pth"):\n            raise ValueError("Use the official ViT-L/16 LVD1689M 8aa4cbdd .pth with its original filename; SAT/plus formats are not supported here")\n        kwargs["weights"] = str(weights)\n    # hubconf also imports unrelated detector/segmentor dependencies. Use the\n    # exact official backbone factory without executing those entry points.\n    package = sys.modules.get("dinov3")\n    if package is not None and not Path(package.__file__).resolve().is_relative_to(repo):\n        raise RuntimeError("Another DINOv3 source is already imported. Restart the kernel before loading this bundle.")\n    sys.path.insert(0, str(repo))\n    backbones = importlib.import_module("dinov3.hub.backbones")\n    if not Path(backbones.__file__).resolve().is_relative_to(repo):\n        raise RuntimeError("DINOv3 backbone code did not load from the requested local repository")\n    backbone = backbones.dinov3_vitl16(**kwargs)\n    if len(backbone.blocks) != 24:\n        raise ValueError("Expected 24 transformer blocks for ViT-L/16")\n    return VisualTACS(backbone, config)\n\n\ndef normalize_map(raw):\n    raw = np.asarray(raw, dtype=np.float32)\n    if raw.ndim != 2 or not np.isfinite(raw).all() or raw.min() < 0:\n        raise ValueError("Invalid attention map")\n    span = float(raw.max() - raw.min())\n    return (raw - raw.min()) / max(span, 1e-12), span > 1e-12\n\n\ndef attention_overlay(image, raw, alpha=0.45):\n    image = read_rgb(image)\n    normalized, informative = normalize_map(raw)\n    heat = Image.fromarray(np.uint8(np.round(normalized * 255))).resize(image.size, Image.Resampling.BILINEAR)\n    values = np.asarray(heat, dtype=np.float32) / 255.0\n    colors = np.stack((values, 0.25 * (1 - values), 1 - values), axis=-1)\n    pixels = np.asarray(image, dtype=np.float32) / 255.0\n    mixed = (1 - alpha) * pixels + alpha * colors\n    # Flat attention is preserved and flagged, never replaced by invented structure.\n    return Image.fromarray(np.uint8(np.clip(mixed * 255, 0, 255))), informative\n\n\ndef png_data_url(image):\n    import base64\n    handle = io.BytesIO()\n    image.save(handle, format="PNG")\n    return "data:image/png;base64," + base64.b64encode(handle.getvalue()).decode("ascii")\n\n\ndef verify_bundle(root):\n    root = Path(root).resolve()\n    manifest = read_json(root / "manifest.json")\n    if manifest.get("schema") != SCHEMA or manifest.get("status") != "ready":\n        raise ValueError("Bundle is incomplete or uses another schema")\n    files = manifest.get("files", {})\n    required = {"model.pt", "config.json", "class_names.json", "candidate_pool.json", "candidate_features.pt",\n                "runtime/tacs_visual_web.py", "vendor/dinov3/dinov3/hub/backbones.py"}\n    if not required.issubset(files):\n        raise ValueError("Manifest omits a required inference artifact")\n    for name, info in files.items():\n        path = bundle_path(root, name)\n        if not path.is_file() or path.stat().st_size != int(info["bytes"]) or file_sha256(path) != info["sha256"]:\n            raise ValueError(f"Bundle checksum mismatch: {name}")\n    return manifest\n\n\nclass BundlePredictor:\n    """Load a trusted, exported bundle. No training Inputs or network are needed."""\n    def __init__(self, root, device="cpu", model_factory=build_model):\n        self.root = Path(root).resolve()\n        self.manifest = verify_bundle(self.root)\n        self.config = read_json(self.root / "config.json")\n        self.class_names = read_json(self.root / "class_names.json")\n        self.candidates = read_json(self.root / "candidate_pool.json")\n        self.device = torch.device(device)\n        if len(self.class_names) != self.config["num_classes"] or len(set(self.class_names)) != len(self.class_names):\n            raise ValueError("Class order/count is invalid")\n        if len(self.candidates) != self.config["candidate_count"]:\n            raise ValueError("Candidate count mismatch")\n        ids = [row["image_id"] for row in self.candidates]\n        if len(set(ids)) != len(ids):\n            raise ValueError("Duplicate candidate IDs")\n        for position, row in enumerate(self.candidates):\n            if row["position"] != position or row["file"] not in self.manifest["files"]:\n                raise ValueError("Candidate bank order or image manifest is invalid")\n            if row["sha256"] != self.manifest["files"][row["file"]]["sha256"]:\n                raise ValueError("Candidate content hash does not match its image")\n            if self.class_names[row["class_id"]] != row["class_name"]:\n                raise ValueError("Candidate class mapping mismatch")\n            bundle_path(self.root, row["file"])\n        checkpoint_state = torch.load(self.root / "model.pt", map_location="cpu", weights_only=True)\n        if checkpoint_state["config"] != self.config or checkpoint_state["class_names"] != self.class_names:\n            raise ValueError("Model/config/class mapping mismatch")\n        bank = torch.load(self.root / "candidate_features.pt", map_location="cpu", weights_only=True)\n        if bank["image_ids"] != ids or bank["checkpoint_sha256"] != file_sha256(self.root / "model.pt"):\n            raise ValueError("Candidate features belong to another checkpoint/order")\n        if tuple(bank["features"].shape) != (len(ids), self.config["feature_dim"]) or not torch.isfinite(bank["features"]).all():\n            raise ValueError("Candidate features are invalid")\n        self.model = model_factory(self.root / "vendor/dinov3", self.config).to(self.device)\n        self.model.load_state_dict(checkpoint_state["model_state_dict"], strict=True)\n        self.model.eval()\n        self.bank = bank["features"].float().to(self.device)\n        self.last_logits = None\n\n    def predict(self, source, top_k=5, explain=True, image_id=None):\n        image = read_rgb(source)\n        query = preprocess(image, self.config).unsqueeze(0).to(self.device)\n        excluded = torch.zeros(len(self.candidates), dtype=torch.bool, device=self.device)\n        if isinstance(source, (str, Path)):\n            content_hash = file_sha256(source)\n        elif isinstance(source, bytes):\n            content_hash = hashlib.sha256(source).hexdigest()\n        else:\n            content_hash = None\n        for i, row in enumerate(self.candidates):\n            if row["image_id"] == image_id or (content_hash is not None and row["sha256"] == content_hash):\n                excluded[i] = True\n        with torch.no_grad():\n            scores, _ = self.model.score(query, self.bank)\n            scores = scores.masked_fill(excluded.unsqueeze(0), -torch.inf)\n            if not torch.isfinite(scores).any():\n                raise ValueError("No eligible candidate after self-exclusion")\n            position = int(scores.argmax(1).item())\n            row = self.candidates[position]\n            candidate = read_rgb(bundle_path(self.root, row["file"]))\n            context = preprocess(candidate, self.config).unsqueeze(0).to(self.device)\n            if explain:\n                logits, maps = self.model(query, context, return_attention=True)\n                _, selector_map = self.model.encode_selector(context, return_attention=True)\n            else:\n                logits = self.model(query, context)\n            self.last_logits = logits.detach().cpu()\n            probabilities = logits.float().softmax(1)[0]\n            values, indices = probabilities.topk(min(max(1, int(top_k)), len(self.class_names)))\n        result = {\n            "model": "tacs", "dataset": self.config["dataset"], "bundle_schema": SCHEMA,\n            "prediction": {"class_id": int(indices[0]), "label": self.class_names[int(indices[0])], "confidence": float(values[0])},\n            "top_k": [{"class_id": int(i), "label": self.class_names[int(i)], "confidence": float(v)} for v, i in zip(values, indices)],\n            "evidence": {\n                "query_caption": {"text": None, "status": "not_used"},\n                "candidate": {"image_id": row["image_id"], "position": position, "image_url": png_data_url(candidate), "caption": None,\n                              "selector_score": float(scores[0, position]), "status": "ready"},\n                "attention": {},\n            },\n        }\n        if explain:\n            for name, raw, source_image in [("query_classifier", maps["query_classifier"][0], image),\n                                            ("candidate_selector", selector_map[0], candidate)]:\n                array = raw.float().cpu().numpy()\n                overlay, informative = attention_overlay(source_image, array)\n                result["evidence"]["attention"][name] = {\n                    "image_url": png_data_url(overlay), "status": "ready" if informative else "flat_attention",\n                    "method": ATTENTION_METHOD, "raw_grid": array.tolist(), "grid_shape": list(array.shape),\n                    "normalization": "per_image_min_max_for_display_only", "class_conditional": False,\n                }\n        return result\n\n\ndef main():\n    import argparse\n    parser = argparse.ArgumentParser(description="Offline visual TACS SUN397 bundle inference")\n    parser.add_argument("--bundle", type=Path, required=True)\n    parser.add_argument("--image", type=Path, required=True)\n    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    parser.add_argument("--output", type=Path, default=Path("prediction.json"))\n    parser.add_argument("--no-attention", action="store_true")\n    args = parser.parse_args()\n    predictor = BundlePredictor(args.bundle, args.device)\n    write_json(args.output, predictor.predict(args.image, explain=not args.no_attention))\n    print(f"Saved prediction and evidence to {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'train_sun397_tacs_web.py': '"""SUN397 visual TACS training and verified, self-contained web export."""\nfrom __future__ import annotations\n\nimport contextlib\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path, PurePosixPath\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport uuid\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom tqdm.auto import tqdm\n\nimport tacs_visual_web as runtime\n\n\ndef default_config():\n    return {\n        "schema": runtime.SCHEMA, "dataset": "SUN397", "method": "TACS_visual",\n        "backbone": "dinov3_vitl16", "feature_dim": 1024, "num_classes": 397,\n        "pretrained_variant": "LVD1689M_8aa4cbdd", "rope_training": "native_per_block_resampling",\n        "image_size": 384, "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],\n        "resize": "PIL_bicubic_stretch_square", "head_dropout": 0.1,\n        "context_top_k": 1, "seed": 42, "val_fraction": 0.1, "pool_fraction": 0.2,\n        "epochs": 100, "batch_size": 2, "effective_batch_size": 64, "workers": 0,\n        "feature_batch_size": 8, "eval_batch_size": 8, "st_bank_chunk": 32, "gradient_checkpointing": True,\n        "backbone_lr": 3e-6, "head_lr": 1e-4, "weight_decay": 0.05,\n        "warmup_epochs": 5, "gumbel_tau": 0.1, "policy_weight": 1.0,\n        "entropy_weight": 0.0, "label_smoothing": 0.0, "grad_clip": 1.0,\n        "utility": "shared_cls_dot_product", "candidate_bank_update": "epoch_start_and_before_validation",\n        "candidate_training_pixels": "cpu_float16_with_exact_st_weight_gradient",\n        "train_augmentation": "PIL_random_resized_crop_scale_0.7_1.0_ratio_0.75_1.333_horizontal_flip_0.5",\n        "inference_precision": "float32", "attention_method": runtime.ATTENTION_METHOD,\n        "caption_used": False, "context_label_prior": False,\n    }\n\n\ndef validate_config(config):\n    if config["backbone"] != "dinov3_vitl16" or config["feature_dim"] != 1024:\n        raise ValueError("This notebook is ViT-L/16 only")\n    if config["num_classes"] != 397 or config["context_top_k"] != 1 or config["caption_used"]:\n        raise ValueError("Expected SUN397 visual TACS, K=1, no captions")\n    if config["image_size"] < 32 or config["image_size"] % 16:\n        raise ValueError("Image size must be a positive multiple of patch size 16")\n    if config["batch_size"] < 2:\n        raise ValueError("Use train batch_size >=2 for within-batch standardized policy rewards")\n    if config["effective_batch_size"] % config["batch_size"]:\n        raise ValueError("effective_batch_size must be divisible by batch_size")\n    if config["epochs"] < 1 or not 0 < config["val_fraction"] < 1 or not 0 < config["pool_fraction"] < 1:\n        raise ValueError("Invalid epoch/split/pool settings")\n    if config["gumbel_tau"] <= 0 or config["st_bank_chunk"] < 1 or config["workers"] < 0:\n        raise ValueError("Invalid Gumbel/loader settings")\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n        torch.backends.cudnn.benchmark = False\n\n\ndef environment():\n    versions = {}\n    for name in ("torch", "numpy", "Pillow", "tqdm", "torchvision", "transformers"):\n        try:\n            versions[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            versions[name] = None\n    return {"python": sys.version, "platform": sys.platform, "packages": versions,\n            "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}\n\n\ndef csv_write(path, rows, fields=None):\n    rows = list(rows)\n    if not rows and fields is None:\n        raise ValueError("Cannot infer columns for an empty CSV")\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields or list(rows[0]), extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(path)\n\n\ndef atomic_torch_save(path, payload):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\n\ndef source_tree_identity(repo):\n    repo = Path(repo)\n    files = {p.relative_to(repo).as_posix(): runtime.file_sha256(p)\n             for p in sorted(repo.rglob("*.py")) if ".git" not in p.parts and "__pycache__" not in p.parts}\n    if "hubconf.py" not in files or not any(p.startswith("dinov3/") for p in files):\n        raise ValueError("Incomplete DINOv3 repository")\n    if not any(p.is_file() and p.name.upper().startswith(("LICENSE", "NOTICE")) for p in repo.iterdir()):\n        raise FileNotFoundError("Attach complete DINOv3 source including original LICENSE/NOTICE files before training")\n    return hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest()\n\n\ndef resolve_weights(reference):\n    path = Path(reference).expanduser()\n    if path.is_file():\n        return path.resolve()\n    if path.is_dir():\n        matches = [p for p in path.rglob("*.pth") if "vitl16" in p.name.lower() and "plus" not in p.name.lower()]\n        if len(matches) == 1:\n            return matches[0].resolve()\n        raise ValueError(f"Expected one ViT-L/16 .pth in {path}; found {len(matches)}. Set the exact filename.")\n    raise FileNotFoundError(f"Attach local DINOv3 ViT-L/16 weights and set DINOV3_WEIGHTS: {path}")\n\n\ndef _relative_id(raw):\n    raw = str(raw).strip().replace("\\\\", "/").lstrip("/")\n    parts = list(PurePosixPath(raw).parts)\n    if parts and parts[0].lower() in {"sun397", "images"}:\n        parts = parts[1:]\n    if len(parts) < 3 or ".." in parts or ":" in raw:\n        raise ValueError(f"Unexpected SUN397 relative path: {raw}")\n    return PurePosixPath(*parts).as_posix()\n\n\ndef load_sun397(dataset_root, train_partition, test_partition, config):\n    root = Path(dataset_root).expanduser().resolve()\n    name_file = next((p for p in (root / "ClassName.txt", root.parent / "ClassName.txt") if p.is_file()), None)\n    if name_file is None:\n        raise FileNotFoundError(f"ClassName.txt not found at {root} or its parent")\n    classes = [line.strip().lstrip("/").replace("\\\\", "/") for line in name_file.read_text(encoding="utf-8-sig").splitlines() if line.strip()]\n    if len(classes) != config["num_classes"] or len(set(classes)) != len(classes):\n        raise ValueError("Expected 397 unique SUN397 classes, preserving ClassName.txt order")\n    class_to_id = {name: i for i, name in enumerate(classes)}\n\n    def partition(path, split):\n        path = Path(path).expanduser().resolve()\n        rows = []\n        for raw in path.read_text(encoding="utf-8-sig").splitlines():\n            if not raw.strip():\n                continue\n            image_id = _relative_id(raw)\n            class_name = str(PurePosixPath(image_id).parent)\n            if class_name not in class_to_id:\n                raise ValueError(f"Unknown class in official partition: {image_id}")\n            image_path = root.joinpath(*PurePosixPath(image_id).parts).resolve()\n            if not image_path.is_relative_to(root) or not image_path.is_file():\n                raise FileNotFoundError(f"Missing SUN397 image: {image_path}")\n            rows.append({"image_id": image_id, "image_path": str(image_path), "class_id": class_to_id[class_name],\n                         "class_name": class_name, "split": split})\n        ids = [row["image_id"] for row in rows]\n        counts = np.bincount([row["class_id"] for row in rows], minlength=len(classes))\n        if len(ids) != len(classes) * 50 or len(set(ids)) != len(ids) or not np.all(counts == 50):\n            raise ValueError(f"{path} must contain 50 distinct images per class (19,850 images)")\n        return rows\n\n    official_train = partition(train_partition, "train")\n    test = partition(test_partition, "test")\n    if {r["image_id"] for r in official_train} & {r["image_id"] for r in test}:\n        raise ValueError("Official train/test image IDs overlap")\n    rng = np.random.default_rng(config["seed"])\n    val_positions = set()\n    for class_id in range(len(classes)):\n        positions = [i for i, row in enumerate(official_train) if row["class_id"] == class_id]\n        count = max(1, min(len(positions) - 1, round(len(positions) * config["val_fraction"])))\n        val_positions.update(int(i) for i in rng.choice(positions, count, replace=False))\n    train, val = [], []\n    for i, row in enumerate(official_train):\n        item = dict(row)\n        item["split"] = "val" if i in val_positions else "train"\n        (val if i in val_positions else train).append(item)\n    count = int(len(train) * config["pool_fraction"])\n    pool_positions = np.random.default_rng(config["seed"]).choice(len(train), count, replace=False)\n    candidates = [dict(train[int(i)], position=position) for position, i in enumerate(pool_positions)]\n    if count < 2 or {r["image_id"] for r in candidates} & {r["image_id"] for r in val + test}:\n        raise ValueError("Candidate pool must be a nonempty subset of actual training images only")\n    return classes, train, val, test, candidates\n\n\ndef augment(image, config):\n    image = runtime.read_rgb(image)\n    width, height = image.size\n    area = width * height\n    box = None\n    for _ in range(10):\n        target = area * random.uniform(0.7, 1.0)\n        ratio = math.exp(random.uniform(math.log(0.75), math.log(4 / 3)))\n        crop_w, crop_h = round(math.sqrt(target * ratio)), round(math.sqrt(target / ratio))\n        if 0 < crop_w <= width and 0 < crop_h <= height:\n            x, y = random.randint(0, width - crop_w), random.randint(0, height - crop_h)\n            box = (x, y, x + crop_w, y + crop_h)\n            break\n    if box is None:\n        side = min(width, height)\n        x, y = (width - side) // 2, (height - side) // 2\n        box = (x, y, x + side, y + side)\n    image = image.crop(box)\n    if random.random() < 0.5:\n        image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n    return runtime.preprocess(image, config)\n\n\nclass QueryDataset(Dataset):\n    def __init__(self, rows, config, training=False):\n        self.rows, self.config, self.training = rows, config, training\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        try:\n            image = augment(row["image_path"], self.config) if self.training else runtime.preprocess(row["image_path"], self.config)\n        except Exception as exc:\n            raise RuntimeError(f"Unreadable image {row[\'image_id\']}: {exc}") from exc\n        return {"image": image, "label": row["class_id"], "image_id": row["image_id"]}\n\n\ndef worker_seed(_worker_id):\n    value = torch.initial_seed() % (2 ** 32)\n    random.seed(value)\n    np.random.seed(value)\n\n\ndef make_loader(rows, config, training=False):\n    kwargs = {"num_workers": config["workers"], "pin_memory": torch.cuda.is_available(), "worker_init_fn": worker_seed}\n    if config["workers"]:\n        kwargs.update(persistent_workers=False, prefetch_factor=2)\n    size = config["batch_size"] if training else config["eval_batch_size"]\n    return DataLoader(QueryDataset(rows, config, training), batch_size=size, shuffle=training,\n                      drop_last=training, generator=torch.Generator().manual_seed(config["seed"]), **kwargs)\n\n\ndef build_image_bank(candidates, config):\n    count, size = len(candidates), config["image_size"]\n    bank = torch.empty((count, 3, size, size), dtype=torch.float16, device="cpu")\n    for index, row in enumerate(tqdm(candidates, desc="Candidate pixels on CPU")):\n        bank[index].copy_(runtime.preprocess(row["image_path"], config))\n    return bank\n\n\n@torch.no_grad()\ndef refresh_bank(model, candidates, config, device, image_bank=None, amp=False):\n    was_training = model.training\n    model.eval()\n    chunks = []\n    for start in range(0, len(candidates), config["feature_batch_size"]):\n        group = candidates[start:start + config["feature_batch_size"]]\n        # Evaluate raw images in fp32; the optional fp16 pixel bank is training-only.\n        images = torch.stack([runtime.preprocess(r["image_path"], config) for r in group]).to(device)\n        with autocast_context(device, amp):\n            features = model.encode_selector(images)\n        chunks.append(features.detach().float().cpu())\n    model.train(was_training)\n    bank = torch.cat(chunks)\n    if not torch.isfinite(bank).all():\n        raise FloatingPointError("Candidate features contain NaN/Inf")\n    return bank\n\n\ndef autocast_context(device, enabled=True):\n    device = torch.device(device)\n    if device.type != "cuda" or not enabled:\n        return contextlib.nullcontext()\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    return torch.autocast("cuda", dtype=dtype)\n\n\ndef classification_metrics(truth, predictions, classes):\n    counts = np.zeros((classes, classes), dtype=np.int64)\n    np.add.at(counts, (np.asarray(truth, dtype=int), np.asarray(predictions, dtype=int)), 1)\n    tp = counts.diagonal()\n    denominator = counts.sum(0) + counts.sum(1)\n    f1 = np.divide(2 * tp, denominator, out=np.zeros(classes, dtype=float), where=denominator != 0)\n    return {"accuracy": float(tp.sum() / max(1, counts.sum())), "macro_f1": float(f1.mean())}\n\n\ndef rng_state(loader):\n    numpy_state = np.random.get_state()\n    return {"python": random.getstate(), "numpy": [numpy_state[0], numpy_state[1].tolist(), numpy_state[2], numpy_state[3], numpy_state[4]],\n            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n            "loader": loader.generator.get_state()}\n\n\ndef restore_rng(saved, loader):\n    random.setstate(saved["python"])\n    state = saved["numpy"]\n    np.random.set_state((state[0], np.asarray(state[1], dtype=np.uint32), state[2], state[3], state[4]))\n    torch.set_rng_state(saved["torch"].cpu())\n    if torch.cuda.is_available() and saved["cuda"]:\n        torch.cuda.set_rng_state_all([s.cpu() for s in saved["cuda"]])\n    loader.generator.set_state(saved["loader"].cpu())\n\n\ndef run_identity(config, classes, frames, assets):\n    ids = {key: [(r["image_id"], int(r["class_id"])) for r in value] for key, value in frames.items()}\n    payload = {"config": config, "classes": classes, "splits_and_pool": ids, "assets": assets,\n               "model_source_sha256": runtime.file_sha256(Path(runtime.__file__)), "training_source_sha256": runtime.file_sha256(__file__)}\n    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\n\n\ndef train_epoch(model, loader, feature_bank, pixel_bank, candidates, optimizer, scaler, config, device, epoch):\n    model.train()\n    bank = feature_bank.to(device)\n    positions = {row["image_id"]: i for i, row in enumerate(candidates)}\n    accumulate = config["effective_batch_size"] // config["batch_size"]\n    stats = {key: 0.0 for key in ("loss", "ce_loss", "policy_loss", "reward", "entropy")}\n    truth, predictions = [], []\n    optimizer_steps = skipped_steps = 0\n    optimizer.zero_grad(set_to_none=True)\n    steps = len(loader)\n    if steps < 1:\n        raise ValueError("Training loader is empty")\n    for step, batch in enumerate(tqdm(loader, desc=f"Epoch {epoch:03d} train")):\n        query, labels = batch["image"].to(device), batch["label"].to(device)\n        excluded = torch.tensor([positions.get(i, -1) for i in batch["image_id"]], device=device)\n        with autocast_context(device):\n            scores, _ = model.score(query, bank, excluded)\n            weights = F.gumbel_softmax(scores, tau=config["gumbel_tau"], hard=True, dim=1)\n            context = runtime.straight_through_images(weights, pixel_bank, config["st_bank_chunk"])\n            logits = model(query, context)\n            ce = F.cross_entropy(logits.float(), labels, label_smoothing=config["label_smoothing"])\n            distribution = torch.distributions.Categorical(logits=scores)\n            actions = distribution.sample()\n            log_probs = distribution.log_prob(actions)\n            entropy = distribution.entropy().mean()\n        # Reward passes use the same eval-mode task network and detached outputs.\n        model.eval()\n        with torch.no_grad(), autocast_context(device):\n            baseline = model(query)\n            reward_context = pixel_bank.index_select(0, actions.cpu()).to(device, dtype=torch.float32)\n            paired = model(query, reward_context)\n            reward = (F.cross_entropy(baseline.float(), labels, reduction="none") -\n                      F.cross_entropy(paired.float(), labels, reduction="none")).detach()\n            advantage = ((reward - reward.mean()) / reward.std(unbiased=False).clamp_min(1e-6)).detach()\n        model.train()\n        policy = -(log_probs * advantage).mean()\n        loss = ce + config["policy_weight"] * policy - config["entropy_weight"] * entropy\n        if not torch.isfinite(loss):\n            raise FloatingPointError(f"Nonfinite loss at epoch {epoch}, batch {step}")\n        group_start = (step // accumulate) * accumulate\n        divisor = min(accumulate, steps - group_start)\n        scaler.scale(loss / divisor).backward()\n        if (step + 1) % accumulate == 0 or step + 1 == steps:\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"], error_if_nonfinite=not scaler.is_enabled())\n            previous_scale = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            skipped = scaler.is_enabled() and scaler.get_scale() < previous_scale\n            skipped_steps += int(skipped)\n            optimizer_steps += int(not skipped)\n            optimizer.zero_grad(set_to_none=True)\n        count = len(labels)\n        for key, value in (("loss", loss), ("ce_loss", ce), ("policy_loss", policy), ("reward", reward.mean()), ("entropy", entropy)):\n            stats[key] += float(value.detach()) * count\n        truth.extend(labels.cpu().tolist())\n        predictions.extend(logits.detach().argmax(1).cpu().tolist())\n    if not optimizer_steps:\n        raise FloatingPointError("Every optimizer step overflowed; inspect AMP/learning rates before resuming")\n    return {**{k: v / len(truth) for k, v in stats.items()}, **classification_metrics(truth, predictions, config["num_classes"]),\n            "optimizer_steps": optimizer_steps, "amp_skipped_steps": skipped_steps}\n\n\n@torch.no_grad()\ndef evaluate(model, loader, bank, pixel_bank, candidates, config, device, label="validation", amp=False):\n    model.eval()\n    features = bank.to(device)\n    positions = {row["image_id"]: i for i, row in enumerate(candidates)}\n    rows, loss_sum = [], 0.0\n    for batch in tqdm(loader, desc=label):\n        query, labels = batch["image"].to(device), batch["label"].to(device)\n        excluded = torch.tensor([positions.get(i, -1) for i in batch["image_id"]], device=device)\n        with autocast_context(device, amp):\n            scores, _ = model.score(query, features, excluded)\n            indices = scores.argmax(1)\n            # Raw files + identical eval transform, matching exported inference.\n            context = torch.stack([runtime.preprocess(candidates[int(i)]["image_path"], config) for i in indices.cpu()]).to(device)\n            logits = model(query, context)\n            loss_sum += float(F.cross_entropy(logits.float(), labels, reduction="sum"))\n        for j, image_id in enumerate(batch["image_id"]):\n            pos = int(indices[j])\n            rows.append({"image_id": image_id, "ground_truth": int(labels[j]), "prediction": int(logits[j].argmax()),\n                         "candidate_position": pos, "candidate_id": candidates[pos]["image_id"]})\n    if not rows:\n        raise ValueError("Evaluation loader is empty")\n    metrics = classification_metrics([r["ground_truth"] for r in rows], [r["prediction"] for r in rows], config["num_classes"])\n    return {**metrics, "loss": loss_sum / len(rows), "images": len(rows)}, rows\n\n\ndef train_run(model, classes, train, val, candidates, config, assets, output, device):\n    validate_config(config)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    identity = run_identity(config, classes, {"train": train, "val": val, "pool": candidates}, assets)\n    protocol = output / "protocol.json"\n    if protocol.is_file() and runtime.read_json(protocol)["identity"] != identity:\n        raise ValueError("Output directory belongs to another run. Do not change seed/epochs/pool/source on resume.")\n    source_hashes = {"runtime": runtime.file_sha256(runtime.__file__), "training": runtime.file_sha256(__file__)}\n    runtime.write_json(protocol, {"identity": identity, "config": config, "assets": assets,\n                                  "source_sha256": source_hashes, "environment": environment()})\n    csv_write(output / "train_split.csv", train)\n    csv_write(output / "validation_split.csv", val)\n    csv_write(output / "candidate_pool.csv", candidates)\n    runtime.write_json(output / "class_names.json", classes)\n    train_loader, val_loader = make_loader(train, config, True), make_loader(val, config)\n    pixel_bank = build_image_bank(candidates, config)\n    optimizer = torch.optim.AdamW([\n        {"params": model.backbone.parameters(), "lr": config["backbone_lr"]},\n        {"params": model.head.parameters(), "lr": config["head_lr"]},\n    ], weight_decay=config["weight_decay"])\n\n    def lr_scale(epoch):\n        warmup = min(config["warmup_epochs"], config["epochs"])\n        if warmup and epoch < warmup:\n            return 0.2 + 0.8 * epoch / warmup\n        progress = (epoch - warmup) / max(1, config["epochs"] - warmup)\n        return 0.01 + 0.99 * (1 + math.cos(math.pi * min(1, progress))) / 2\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_scale)\n    scaler = torch.amp.GradScaler("cuda", enabled=torch.device(device).type == "cuda" and not torch.cuda.is_bf16_supported())\n    last_path, best_path = output / "last.pt", output / "best.pt"\n    history, best_loss, start_epoch = [], math.inf, 1\n    if last_path.is_file():\n        last = torch.load(last_path, map_location="cpu", weights_only=True)\n        if last["identity"] != identity:\n            raise ValueError("Resume identity mismatch")\n        model.load_state_dict(last["model_state_dict"], strict=True)\n        optimizer.load_state_dict(last["optimizer_state_dict"])\n        scheduler.load_state_dict(last["scheduler_state_dict"])\n        scaler.load_state_dict(last["scaler_state_dict"])\n        restore_rng(last["rng"], train_loader)\n        history, best_loss, start_epoch = last["history"], last["best_val_loss"], last["epoch"] + 1\n        del last\n        if not best_path.is_file():\n            raise FileNotFoundError("Resume requires this run\'s best.pt as well as last.pt")\n        print(f"Resuming after completed epoch {start_epoch - 1}; horizon stays {config[\'epochs\']}")\n    for epoch in range(start_epoch, config["epochs"] + 1):\n        started = time.perf_counter()\n        bank = refresh_bank(model, candidates, config, device, amp=True)\n        train_metrics = train_epoch(model, train_loader, bank, pixel_bank, candidates, optimizer, scaler, config, device, epoch)\n        bank = refresh_bank(model, candidates, config, device, amp=True)\n        val_metrics, _ = evaluate(model, val_loader, bank, pixel_bank, candidates, config, device, amp=True)\n        improved = val_metrics["loss"] < best_loss\n        best_loss = min(best_loss, val_metrics["loss"])\n        history.append({"epoch": epoch, **{"train_" + k: v for k, v in train_metrics.items()},\n                        **{"val_" + k: v for k, v in val_metrics.items()}, "seconds": time.perf_counter() - started,\n                        "backbone_lr": optimizer.param_groups[0]["lr"]})\n        scheduler.step()\n        model_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}\n        base = {"schema": runtime.SCHEMA, "identity": identity, "epoch": epoch, "model_state_dict": model_state,\n                "config": config, "class_names": classes, "val_metrics": val_metrics}\n        if improved:\n            atomic_torch_save(best_path, base)\n        atomic_torch_save(last_path, {**base, "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(),\n                                     "scaler_state_dict": scaler.state_dict(), "rng": rng_state(train_loader),\n                                     "history": history, "best_val_loss": best_loss})\n        csv_write(output / "history.csv", history)\n        print(f"epoch={epoch:03d}/{config[\'epochs\']} val_loss={val_metrics[\'loss\']:.6f} val_f1={val_metrics[\'macro_f1\']:.6f} best={improved}")\n        del model_state, base\n    if not history or history[-1]["epoch"] != config["epochs"]:\n        raise RuntimeError("Training budget did not complete; final test/export are blocked")\n    best = torch.load(best_path, map_location="cpu", weights_only=True)\n    if best["identity"] != identity:\n        raise ValueError("Best checkpoint belongs to another run")\n    model.load_state_dict(best["model_state_dict"], strict=True)\n    model.eval()\n    runtime.write_json(output / "training_complete.json", {"identity": identity, "epochs_completed": config["epochs"],\n                                                         "best_epoch": best["epoch"], "best_sha256": runtime.file_sha256(best_path)})\n    return best, history\n\n\ndef copy_vendor(repo, destination):\n    repo, destination = Path(repo), Path(destination)\n    package = repo / "dinov3"\n    if not package.is_dir() or not (repo / "hubconf.py").is_file():\n        raise FileNotFoundError("Cannot export incomplete DINOv3 source")\n    ignore = shutil.ignore_patterns("__pycache__", "*.pyc", ".git", "*.pth", "*.pt", "*.safetensors", ".cache")\n    destination.mkdir(parents=True, exist_ok=False)\n    # Backbone inference needs these upstream modules, not long training YAML\n    # trees or detection/segmentation imports. Preserve selected sources verbatim.\n    vendored = destination / "dinov3"\n    vendored.mkdir()\n    shutil.copy2(package / "__init__.py", vendored / "__init__.py")\n    for name in ("models", "layers", "utils"):\n        shutil.copytree(package / name, vendored / name, ignore=ignore)\n    (vendored / "hub").mkdir()\n    for name in ("__init__.py", "backbones.py", "utils.py"):\n        shutil.copy2(package / "hub" / name, vendored / "hub" / name)\n    licenses = [p for p in repo.iterdir() if p.is_file() and (p.name.upper().startswith(("LICENSE", "NOTICE")))]\n    if not licenses:\n        raise FileNotFoundError("Attach the original DINOv3 source with its license/notice files")\n    for path in licenses:\n        shutil.copy2(path, destination / path.name)\n\n\ndef export_bundle(model, best, candidates, classes, config, repo, output, val_example, test_metrics, test_predictions,\n                  model_factory=runtime.build_model, require_training_complete=True):\n    """Only call after loading best.pt; always rebuild the final feature bank."""\n    output = Path(output).resolve()\n    if require_training_complete:\n        complete = runtime.read_json(output / "training_complete.json")\n        if complete["identity"] != best["identity"] or complete["best_epoch"] != best["epoch"]:\n            raise ValueError("Training/best/export identity mismatch")\n        if complete["best_sha256"] != runtime.file_sha256(output / "best.pt"):\n            raise ValueError("Best checkpoint changed after completion")\n    if classes != best["class_names"] or config != best["config"]:\n        raise ValueError("Export arguments do not match the best checkpoint")\n    protocol = runtime.read_json(output / "protocol.json")\n    if protocol["identity"] != best["identity"] or protocol["config"] != config:\n        raise ValueError("Export protocol does not match the trained run")\n    source_hashes = {"runtime": runtime.file_sha256(runtime.__file__), "training": runtime.file_sha256(__file__)}\n    if protocol["source_sha256"] != source_hashes:\n        raise ValueError("Runtime/training source changed after training")\n    expected_source = protocol["assets"].get("dino_source_sha256")\n    if expected_source is not None and source_tree_identity(repo) != expected_source:\n        raise ValueError("Export DINOv3 source differs from the training source")\n    with (output / "candidate_pool.csv").open(encoding="utf-8", newline="") as handle:\n        trained_pool = [(r["image_id"], int(r["class_id"])) for r in csv.DictReader(handle)]\n    if trained_pool != [(r["image_id"], int(r["class_id"])) for r in candidates]:\n        raise ValueError("Export candidate pool/order differs from the trained pool")\n    model.load_state_dict(best["model_state_dict"], strict=True)\n    model.eval()\n    device = next(model.parameters()).device\n    export_root = output / "exports"\n    export_root.mkdir(parents=True, exist_ok=True)\n    target = export_root / "sun397_tacs_visual_vitl16_web_bundle"\n    archive = target.with_suffix(".zip")\n    best_sha = runtime.file_sha256(output / "best.pt")\n    if target.exists() or archive.exists():\n        if target.is_dir() and archive.is_file() and (export_root / "download_manifest.json").is_file():\n            saved_manifest = runtime.verify_bundle(target)\n            download = runtime.read_json(export_root / "download_manifest.json")\n            if (saved_manifest["run_identity"] == best["identity"]\n                    and saved_manifest.get("training_checkpoint_sha256") == best_sha\n                    and download["zip"] == archive.name and download["bytes"] == archive.stat().st_size\n                    and download["sha256"] == runtime.file_sha256(archive)):\n                print(f"Reusing this run\'s verified deployment ZIP: {archive}")\n                return archive\n        raise FileExistsError("Export target exists but cannot be verified as this run\'s complete bundle. Preserve it and inspect exports/.")\n    parameter_bytes = sum(v.numel() * v.element_size() for v in model.state_dict().values())\n    image_bytes = sum(Path(row["image_path"]).stat().st_size for row in candidates)\n    if shutil.disk_usage(export_root).free < 2 * (parameter_bytes + image_bytes) + 1024 ** 3:\n        raise OSError("Not enough disk space for bundle plus ZIP; preserve/move last.pt before exporting")\n    stage = export_root / (".staging_" + uuid.uuid4().hex)\n    stage.mkdir()\n    portable_config = dict(config, candidate_count=len(candidates), inference_precision="float32")\n    runtime.write_json(stage / "config.json", portable_config)\n    runtime.write_json(stage / "class_names.json", classes)\n    atomic_torch_save(stage / "model.pt", {"schema": runtime.SCHEMA, "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},\n                                          "class_names": classes, "config": portable_config, "epoch": best["epoch"], "run_identity": best["identity"]})\n    checkpoint_sha = runtime.file_sha256(stage / "model.pt")\n    pool = []\n    for position, row in enumerate(tqdm(candidates, desc="Bundle candidate images")):\n        name = "candidates/" + row["image_id"]\n        destination = runtime.bundle_path(stage, name)\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(row["image_path"], destination)\n        pool.append({"position": position, "image_id": row["image_id"], "file": name, "class_id": int(row["class_id"]),\n                     "class_name": classes[int(row["class_id"])], "sha256": runtime.file_sha256(destination)})\n    runtime.write_json(stage / "candidate_pool.json", pool)\n    csv_write(stage / "candidate_pool.csv", pool)\n    final_bank = refresh_bank(model, candidates, config, device, amp=False)\n    atomic_torch_save(stage / "candidate_features.pt", {"features": final_bank, "image_ids": [r["image_id"] for r in pool],\n                                                      "checkpoint_sha256": checkpoint_sha, "precision": "float32"})\n    copy_vendor(repo, stage / "vendor/dinov3")\n    (stage / "runtime").mkdir()\n    shutil.copy2(Path(runtime.__file__), stage / "runtime/tacs_visual_web.py")\n    (stage / "provenance").mkdir()\n    shutil.copy2(Path(__file__), stage / "provenance/training_source.py")\n    runtime.write_json(stage / "environment.json", environment())\n    runtime.write_json(stage / "metrics.json", {"test": test_metrics, "validation_best": best["val_metrics"], "best_epoch": best["epoch"],\n                                              "run_identity": best["identity"], "test_precision": "float32"})\n    csv_write(stage / "test_predictions.csv", test_predictions)\n    if (output / "history.csv").is_file():\n        shutil.copy2(output / "history.csv", stage / "history.csv")\n    shutil.copy2(output / "protocol.json", stage / "protocol.json")\n    for name in ("train_split.csv", "validation_split.csv", "test_split.csv"):\n        if (output / name).is_file():\n            shutil.copy2(output / name, stage / "provenance" / name)\n    (stage / "examples").mkdir()\n    example_path = stage / "examples" / ("query" + Path(val_example["image_path"]).suffix.lower())\n    shutil.copy2(val_example["image_path"], example_path)\n    runtime.write_json(stage / "attention_config.json", {\n        "method": runtime.ATTENTION_METHOD, "layer": "last_transformer_block", "heads": "mean",\n        "query_classifier": "query patch slice of pair-input classifier CLS attention",\n        "candidate_selector": "selected candidate standalone selector-encoder CLS attention",\n        "raw_maps_included": True, "class_conditional": False, "causal_explanation": False,\n        "no_context_comparison_model_included": False,\n        "note": "No separate attention network is needed; maps are recomputed by the trained backbone. Not an exact replication of every Figure 4 visualization choice.",\n    })\n    versions = environment()["packages"]\n    requirements = [f"{name}=={versions[name].split(\'+\')[0]}" for name in ("torch", "numpy", "Pillow", "tqdm")]\n    (stage / "requirements.txt").write_text("\\n".join(requirements) + "\\n", encoding="utf-8")\n    (stage / "runtime/verify_export.py").write_text(COLD_VERIFY_SOURCE, encoding="utf-8")\n    (stage / "README.md").write_text(\n        "# SUN397 visual TACS ViT-L/16 web bundle\\n\\n"\n        "Full trained weights, exact class order, candidate images and bank, vendored DINOv3 backbone code and runtime.\\n"\n        "No SUN397 training mount or pretrained weight file is needed after extraction.\\n"\n        "Install matching PyTorch for your OS/CUDA and the listed dependencies in a separate environment.\\n"\n        "No Florence, CLIP or captions are used in this visual bundle.\\n\\n"\n        "```bash\\npython runtime/tacs_visual_web.py --bundle . --image examples/" + example_path.name + " --output prediction.json\\n```\\n\\n"\n        "For FastAPI, import BundlePredictor from runtime/tacs_visual_web.py, load once and call predict(file_bytes).\\n"\n        "It returns model/prediction/top_k and evidence.candidate plus query_classifier/candidate_selector PNG data URLs.\\n"\n        "Do not call the CPU/GPU predictor concurrently without a lock. API wiring in the existing web is a separate step.\\n"\n        "Attention is last-block mean-head CLS attention, not Grad-CAM or a class-conditioned causal claim.\\n"\n        "There is no separately trained no-context model, text captioner or textual feature naming in this bundle.\\n"\n        "Keep original DINOv3/model/dataset license terms; do not assume public redistribution rights.\\n",\n        encoding="utf-8")\n    # The manifest is refreshed after offline reload/parity and explanation checks.\n    def manifest():\n        files = {p.relative_to(stage).as_posix(): {"bytes": p.stat().st_size, "sha256": runtime.file_sha256(p)}\n                 for p in sorted(stage.rglob("*")) if p.is_file() and p.name != "manifest.json" and "__pycache__" not in p.parts}\n        runtime.write_json(stage / "manifest.json", {"schema": runtime.SCHEMA, "status": "ready", "files": files,\n                                                   "run_identity": best["identity"], "model_sha256": checkpoint_sha,\n                                                   "training_checkpoint_sha256": best_sha})\n    manifest()\n    cold_reload = model_factory is runtime.build_model\n    if cold_reload:\n        env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")\n        env.pop("PYTHONPATH", None)\n        process = subprocess.run([sys.executable, str(stage / "runtime/verify_export.py"), "--bundle", str(stage),\n                                  "--image", str(example_path), "--image-id", val_example["image_id"], "--device", str(device)],\n                                 cwd=stage, env=env, text=True, capture_output=True)\n        if process.returncode:\n            raise RuntimeError("Cold offline bundle reload failed:\\n" + process.stdout[-4000:] + process.stderr[-8000:])\n        result = runtime.read_json(stage / "examples/prediction.json")\n        reloaded_logits = torch.load(stage / "examples/reload_logits.pt", map_location="cpu", weights_only=True)\n    else:\n        # Dependency injection is used by CPU unit tests, never the notebook\'s production export.\n        predictor = runtime.BundlePredictor(stage, device=str(device), model_factory=model_factory)\n        result = predictor.predict(example_path, explain=True, image_id=val_example["image_id"])\n        reloaded_logits = predictor.last_logits\n        del predictor\n    position = result["evidence"]["candidate"]["position"]\n    with torch.no_grad():\n        query = runtime.preprocess(example_path, config)[None].to(device)\n        reference_scores, _ = model.score(query, final_bank.to(device))\n        example_sha = runtime.file_sha256(example_path)\n        for i, row in enumerate(pool):\n            if row["sha256"] == example_sha or row["image_id"] == val_example["image_id"]:\n                reference_scores[:, i] = -torch.inf\n        if int(reference_scores.argmax(1)) != position:\n            raise ValueError("Export reload selected another candidate than the best checkpoint")\n        reference = model(query, runtime.preprocess(candidates[position]["image_path"], config)[None].to(device)).float().cpu()\n    error = float((reference - reloaded_logits).abs().max())\n    if not torch.allclose(reference, reloaded_logits, atol=1e-4, rtol=1e-4):\n        raise ValueError(f"Export reload logits disagree with best checkpoint: max_abs={error}")\n    for role, item in result["evidence"]["attention"].items():\n        if item["method"] != runtime.ATTENTION_METHOD or not np.isfinite(np.asarray(item["raw_grid"])).all():\n            raise ValueError(f"Invalid exported attention: {role}")\n    runtime.write_json(stage / "examples/prediction.json", result)\n    runtime.write_json(stage / "smoke_test.json", {"passed": True, "offline_local_reload": True, "cold_subprocess_reload": cold_reload,\n                                                  "strict_state_dict": True,\n                                                  "max_abs_logit_error": error, "query_image_id": val_example["image_id"],\n                                                  "selected_candidate_id": result["evidence"]["candidate"]["image_id"],\n                                                  "attention_roles": list(result["evidence"]["attention"]),\n                                                  "flat_attention_roles": [k for k, v in result["evidence"]["attention"].items() if v["status"] != "ready"]})\n    manifest()\n    runtime.verify_bundle(stage)\n    stage.rename(target)\n    temporary = archive.with_name(archive.name + ".tmp")\n    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as handle:\n        for path in sorted(target.rglob("*")):\n            if path.is_file() and "__pycache__" not in path.parts:\n                handle.write(path, arcname=target.name + "/" + path.relative_to(target).as_posix())\n    with zipfile.ZipFile(temporary) as handle:\n        bad = handle.testzip()\n        if bad:\n            raise ValueError(f"Corrupt ZIP member: {bad}")\n    temporary.replace(archive)\n    runtime.write_json(export_root / "download_manifest.json", {"zip": archive.name, "bytes": archive.stat().st_size,\n                                                              "sha256": runtime.file_sha256(archive), "bundle": target.name,\n                                                              "model": config["backbone"], "schema": runtime.SCHEMA})\n    print(f"Verified deployment ZIP: {archive}")\n    return archive\n\n\nCOLD_VERIFY_SOURCE = \'\'\'"""Cold-process inference check using only extracted bundle assets."""\nimport argparse\nimport sys\nfrom pathlib import Path\nimport torch\nfrom tacs_visual_web import BundlePredictor, write_json\n\nparser = argparse.ArgumentParser()\nparser.add_argument("--bundle", type=Path, required=True)\nparser.add_argument("--image", type=Path, required=True)\nparser.add_argument("--image-id", required=True)\nparser.add_argument("--device", required=True)\nargs = parser.parse_args()\nif torch.cuda.is_available():\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\npredictor = BundlePredictor(args.bundle, args.device)\nimport dinov3\nif not Path(dinov3.__file__).resolve().is_relative_to((args.bundle / "vendor/dinov3").resolve()):\n    raise RuntimeError("Reload used an external DINOv3 source instead of the bundle")\nplain = predictor.predict(args.image, explain=False, image_id=args.image_id)\nplain_logits = predictor.last_logits.clone()\nexplained = predictor.predict(args.image, explain=True, image_id=args.image_id)\nif plain["evidence"]["candidate"]["image_id"] != explained["evidence"]["candidate"]["image_id"]:\n    raise RuntimeError("Explanation changed context selection")\nif not torch.allclose(plain_logits, predictor.last_logits, atol=1e-5, rtol=1e-5):\n    raise RuntimeError("Attention capture changed model logits")\nwrite_json(args.bundle / "examples/prediction.json", explained)\ntorch.save(predictor.last_logits, args.bundle / "examples/reload_logits.pt")\nprint("Cold bundle inference, both real attention maps and logits parity passed")\n\'\'\'\n'}
for filename, source in EMBEDDED_SOURCES.items():
    path = RUNTIME_DIR / filename
    if path.is_file() and path.read_text(encoding="utf-8") != source:
        raise ValueError("Runtime source changed in this output directory; preserve it and start a new run")
    path.write_text(source, encoding="utf-8")
sys.path.insert(0, str(RUNTIME_DIR.resolve()))
import tacs_visual_web as web_runtime
import train_sun397_tacs_web as workflow
if Path(web_runtime.__file__).resolve() != (RUNTIME_DIR / "tacs_visual_web.py").resolve():
    raise RuntimeError("Another runtime is already imported. Restart the kernel and Run All.")
CONFIG = workflow.default_config()
CONFIG.update(epochs=EPOCHS, batch_size=TRAIN_BATCH_SIZE, effective_batch_size=EFFECTIVE_BATCH_SIZE,
              workers=NUM_WORKERS, seed=SEED, image_size=IMAGE_SIZE,
              feature_batch_size=16 if GPU_GIB >= 40 else 8, eval_batch_size=8 if GPU_GIB >= 40 else 4)
workflow.validate_config(CONFIG)
workflow.seed_all(SEED)
display(CONFIG)


## 4 Validate local assets and official SUN397 splits

Expected partition 01: 19,850 official train and 19,850 test. Hold out 1,985 stratified validation images from official train, retain 17,865 training queries; a fixed 3,573-image pool is sampled from actual train only. Pool images can also be training queries, with exact self-ID masking. ClassName.txt order is authoritative. Missing images stop the run instead of changing the dataset silently.


In [ ]:
for path in (DATASET_ROOT, TRAIN_PARTITION, TEST_PARTITION, DINOV3_REPO_DIR):
    if not path.exists():
        raise FileNotFoundError(f"Required Input not found: {path}. Correct section 2 before proceeding.")
WEIGHTS_FILE = workflow.resolve_weights(DINOV3_WEIGHTS)
ASSETS = {
    "pretrained_weights_sha256": web_runtime.file_sha256(WEIGHTS_FILE),
    "dino_source_sha256": workflow.source_tree_identity(DINOV3_REPO_DIR),
    "train_partition_sha256": web_runtime.file_sha256(TRAIN_PARTITION),
    "test_partition_sha256": web_runtime.file_sha256(TEST_PARTITION),
}
class_names, train_rows, val_rows, test_rows, candidate_rows = workflow.load_sun397(
    DATASET_ROOT, TRAIN_PARTITION, TEST_PARTITION, CONFIG)
print("Train / val / test / pool:", len(train_rows), len(val_rows), len(test_rows), len(candidate_rows))
print("Backbone weights:", WEIGHTS_FILE)
assert len(class_names) == 397
workflow.csv_write(OUTPUT_DIR / "test_split.csv", test_rows)
web_runtime.write_json(OUTPUT_DIR / "environment.json", workflow.environment())


## 5 Construct L16 and check token positions and real attention

Check solo selector features against the official DINOv3 forward_features implementation. Check pair logits with and without attention capture. For 384/16, both raw maps must have a 24 by 24 grid. These operations do not update weights. No S/16 fallback, missing-RoPE fallback or feature-averaging fallback is permitted.


In [ ]:
model = web_runtime.build_model(DINOV3_REPO_DIR, CONFIG, WEIGHTS_FILE).to(DEVICE)
model.eval()
probe = torch.stack([web_runtime.preprocess(row["image_path"], CONFIG) for row in candidate_rows[:2]]).to(DEVICE)
with torch.no_grad():
    official_features = model.backbone.forward_features(probe)["x_norm_clstoken"]
    selector_features = model.encode_selector(probe)
    if not torch.allclose(official_features, selector_features, atol=1e-4, rtol=1e-4):
        raise RuntimeError("Selector token/RoPE path does not match official DINOv3")
    plain = model(probe[:1], probe[1:])
    explained, pair_maps = model(probe[:1], probe[1:], return_attention=True)
    _, selector_map = model.encode_selector(probe[1:], return_attention=True)
    if not torch.allclose(plain, explained, atol=1e-5, rtol=1e-5):
        raise RuntimeError("Capturing attention changed classifier logits")
    expected_grid = (IMAGE_SIZE // 16, IMAGE_SIZE // 16)
    assert tuple(pair_maps["query_classifier"].shape[-2:]) == expected_grid
    assert tuple(selector_map.shape[-2:]) == expected_grid
    assert torch.isfinite(plain).all() and torch.isfinite(selector_map).all()
    scores, _ = model.score(probe, selector_features, excluded_positions=[0, 1])
    assert scores.argmax(1).tolist() == [1, 0]
print("L/16 dimensions:", model.backbone.embed_dim, "| blocks:", len(model.backbone.blocks))
print("Real attention method:", web_runtime.ATTENTION_METHOD, "| grid:", expected_grid)
del probe, official_features, selector_features, plain, explained, pair_maps, selector_map, scores
torch.cuda.empty_cache()
workflow.seed_all(SEED)


## 6 Train and resume only this run

Both task straight-through gradients and categorical policy gradients update the shared selector backbone. CPU pixel banking uses a chunked exact derivative of hard selection weights multiplied by the fixed pixel bank; it is not an index-only estimator. Policy rewards compare detached query-only and paired CE under the same eval-mode task network. No separate no-context classifier is trained.

Best uses minimum validation CE, not test accuracy. Last checkpoint includes optimizer, scheduler, scaler, RNG and loader generator states. Interrupts preserve the last completed epoch; an interrupted partial epoch is replayed. The final incomplete accumulation group is scaled correctly. With multiple workers, exact augmentation replay is environment-dependent; default workers=0 is simplest for resume.

A new Kaggle session may have empty /kaggle/working. Attach the previous saved Output and restore the full run folder to OUTPUT_DIR, including last.pt, best.pt, protocol.json and embedded_runtime, before resuming. The deployment ZIP alone is not a training-resume checkpoint.


In [ ]:
if RUN_TRAINING:
    BEST, HISTORY = workflow.train_run(model, class_names, train_rows, val_rows, candidate_rows,
                                       CONFIG, ASSETS, OUTPUT_DIR, DEVICE)
else:
    completion = web_runtime.read_json(OUTPUT_DIR / "training_complete.json")
    BEST = torch.load(OUTPUT_DIR / "best.pt", map_location="cpu", weights_only=True)
    expected = workflow.run_identity(CONFIG, class_names, {"train": train_rows, "val": val_rows, "pool": candidate_rows}, ASSETS)
    if completion["identity"] != expected or BEST["identity"] != expected:
        raise ValueError("Completed run does not match this notebook/configuration")
    model.load_state_dict(BEST["model_state_dict"], strict=True)
    model.eval()
print("Best validation epoch:", BEST["epoch"], "| validation:", BEST["val_metrics"])


## 7 Locked final test from best weights

Rebuild candidate features after reloading best, then use float32 raw-image evaluation exactly as the portable runtime does. This final test can differ slightly from AMP validation. Export records precision explicitly. Candidate labels are diagnostics only, never selector/classifier inputs.


In [ ]:
if not RUN_FINAL_TEST:
    raise RuntimeError("Set RUN_FINAL_TEST=True to produce the evaluated deployment bundle")
completion = web_runtime.read_json(OUTPUT_DIR / "training_complete.json")
if completion["identity"] != BEST["identity"] or completion["epochs_completed"] != CONFIG["epochs"]:
    raise RuntimeError("Complete this run's training before final test/export")
model.load_state_dict(BEST["model_state_dict"], strict=True)
model.eval()
FINAL_BANK = workflow.refresh_bank(model, candidate_rows, CONFIG, DEVICE, amp=False)
TEST_METRICS, TEST_PREDICTIONS = workflow.evaluate(
    model, workflow.make_loader(test_rows, CONFIG), FINAL_BANK, None, candidate_rows, CONFIG, DEVICE,
    label="Official SUN397 test / best checkpoint / fp32", amp=False)
web_runtime.write_json(OUTPUT_DIR / "test_metrics.json", {**TEST_METRICS, "best_epoch": BEST["epoch"], "precision": "float32"})
workflow.csv_write(OUTPUT_DIR / "test_predictions.csv", TEST_PREDICTIONS)
display(TEST_METRICS)
del FINAL_BANK
torch.cuda.empty_cache()


## 8 Export complete bundle and verify a cold offline reload

The export copies every candidate image, the exact source and license notices, full best weights and aligned bank. A new Python process loads only the staged bundle (pretrained=False), checks that DINOv3 came from vendor/, predicts a held-out validation example, computes both real attention maps and confirms attention does not alter logits. The notebook compares its own best-model selection/logits to the reloaded bundle, verifies file hashes and ZIP CRC before publishing the ZIP.

Keep `last.pt` for training resume. It is intentionally not inside the deployment ZIP; neither optimizer state nor a second pretrained DINO file is needed for inference. Rerunning this cell reuses a complete ZIP only after verifying its hashes and binding to this run's best checkpoint. A partial or mismatched export is rejected and preserved for inspection.


In [ ]:
if RUN_EXPORT:
    EXPORT_ZIP = workflow.export_bundle(model, BEST, candidate_rows, class_names, CONFIG, DINOV3_REPO_DIR,
                                        OUTPUT_DIR, val_rows[0], TEST_METRICS, TEST_PREDICTIONS)
    print("Download this complete ZIP:", EXPORT_ZIP)
else:
    raise RuntimeError("Training finished, but no deploy bundle was created. Set RUN_EXPORT=True and run this section.")


## 9 Inspect actual query and candidate evidence and download

These are generated by the reloaded trained model, not placeholder maps. A flat attention map is flagged instead of replaced. Attention highlights locations, not textual names of features. This is one shared trained network with two attention readouts, not an additional model that needs separate training.


In [ ]:
import base64
from IPython.display import FileLink, Image as DisplayImage, display
bundle_dir = EXPORT_ZIP.with_suffix("")
smoke = web_runtime.read_json(bundle_dir / "smoke_test.json")
result = web_runtime.read_json(bundle_dir / "examples/prediction.json")
display(smoke)
display(result["prediction"])
for role in ("query_classifier", "candidate_selector"):
    item = result["evidence"]["attention"][role]
    print(role, "|", item["status"], "|", item["method"])
    display(DisplayImage(data=base64.b64decode(item["image_url"].split(",", 1)[1])))
display(FileLink(str(EXPORT_ZIP)))
print("Also available in Kaggle Output under:", EXPORT_ZIP.relative_to(PROJECT_DIR) if EXPORT_ZIP.is_relative_to(PROJECT_DIR) else EXPORT_ZIP)
print("Next step: connect BundlePredictor to the existing FastAPI TACS page; M-TACS is notebook 2, not part of this run.")
